# Population Health Analytics

This notebook evaluates population-level disease burden, healthcare utilization, and patient risk patterns using synthetic clinical and claims data.

The analysis is designed to support:

- population health management,
- care-management prioritization,
- chronic disease intervention planning,
- utilization management,
- and healthcare resource allocation.

It Answers:

- Top Chronic Diseases
- Disease Burden
- High Utilizers
- High Risk Patients

## Imports

In [4]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import Markdown, display



## Configure the project path

In [5]:
PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"

## Import project functions

In [6]:
from src.data_loader import load_tables
from src.claims_analytics import calculate_patient_costs
from src.population_health import (
    calculate_condition_prevalence,
    calculate_patient_condition_burden,
    calculate_patient_utilization,
    build_patient_population_profile,
    stratify_population_risk,
)

In [7]:
from src.claims_analytics import (
    prepare_claim_transactions,
    build_claim_financials,
    calculate_patient_costs,
)

## Load the required tables

In [12]:
tables = load_tables()

patients = tables["patients"].copy()
conditions = tables["conditions"].copy()
encounters = tables["encounters"].copy()

In [13]:
claims = tables["claims"].copy()
claims_transactions = tables["claims_transactions"].copy()


## Confirm the available data

In [14]:
for table_name, dataframe in {
    "patients": patients,
    "conditions": conditions,
    "encounters": encounters,
}.items():
    print(
        f"{table_name.title():<12} "
        f"{len(dataframe):>8,} rows | "
        f"{dataframe.shape[1]:>3} columns"
    )

Patients           16 rows |  28 columns
Conditions        724 rows |   7 columns
Encounters      1,494 rows |  15 columns


## Section 1 — Population Overview

In [15]:
condition_prevalence = calculate_condition_prevalence(
    conditions,
    total_patients=len(patients)
)

patient_condition_burden = calculate_patient_condition_burden(
    conditions
)

patient_utilization = calculate_patient_utilization(
    encounters
)

## Section 2 — Condition Burden.

In [16]:
top_conditions = condition_prevalence.head(10).copy()

top_conditions_display = top_conditions[
    [
        "DESCRIPTION",
        "Patients_With_Condition",
        "Condition_Records",
        "Prevalence_Rate",
    ]
].copy()

top_conditions_display["Prevalence_Rate"] = (
    top_conditions_display["Prevalence_Rate"]
    .map(lambda value: f"{value:.1%}")
)

display(top_conditions_display)

,DESCRIPTION,Patients_With_Condition,Condition_Records,Prevalence_Rate
0,Medication review due (situation),16,109,100.0%
1,Stress (finding),15,56,93.8%
2,Full-time employment (finding),15,55,93.8%
3,Gingivitis (disorder),14,53,87.5%
4,Social isolation (finding),14,31,87.5%
5,Part-time employment (finding),12,33,75.0%
6,Viral sinusitis (disorder),11,17,68.8%
7,Body mass index 30+ - obesity (finding),11,11,68.8%
8,Educated to high school level (finding),11,11,68.8%
9,Victim of intimate partner abuse (finding),10,18,62.5%


### Visualization

In [ ]:
fig = px.bar(
    top_conditions.sort_values(
        "Patients_With_Condition",
        ascending=True,
    ),
    x="Patients_With_Condition",
    y="DESCRIPTION",
    orientation="h",
    title="Top Conditions by Number of Affected Patients",
    labels={
        "Patients_With_Condition": "Unique Patients",
        "DESCRIPTION": "Condition",
    },
    text="Patients_With_Condition",
)

fig.update_layout(
    xaxis_title="Number of Unique Patients",
    yaxis_title="Condition",
    showlegend=False,
)

fig.show()

### Dynamic Consultant Interpretation

In [ ]:
from IPython.display import Markdown, display

top_condition = top_conditions.iloc[0]
second_condition = top_conditions.iloc[1]

display(
    Markdown(
f"""
### Consultant Interpretation

The analysis identified **{top_condition['DESCRIPTION']}** as the most prevalent condition,
affecting **{top_condition['Patients_With_Condition']} of {len(patients)} patients**
(**{top_condition['Prevalence_Rate']:.1%}**) with **{top_condition['Condition_Records']}**
documented records. This suggests frequent documentation of this clinical finding across the patient population.

The second most common condition was **{second_condition['DESCRIPTION']}**, affecting
**{second_condition['Patients_With_Condition']} patients ({second_condition['Prevalence_Rate']:.1%})**.

Several of the most prevalent findings represent **social determinants of health (SDOH)** or
clinical observations (e.g., stress, employment status, social isolation) rather than traditional
chronic diseases. This highlights that the synthetic dataset captures broader patient context in
addition to diagnoses, emphasizing factors that may influence healthcare utilization and population
health management.

From a population health perspective, understanding both medical conditions and social risk factors
can help identify opportunities for preventive interventions, care coordination, and targeted
community support programs.
"""
    )
)


### Consultant Interpretation

The analysis identified **Medication review due (situation)** as the most prevalent condition,
affecting **16 of 16 patients**
(**100.0%**) with **109**
documented records. This suggests frequent documentation of this clinical finding across the patient population.

The second most common condition was **Stress (finding)**, affecting
**15 patients (93.8%)**.

Several of the most prevalent findings represent **social determinants of health (SDOH)** or
clinical observations (e.g., stress, employment status, social isolation) rather than traditional
chronic diseases. This highlights that the synthetic dataset captures broader patient context in
addition to diagnoses, emphasizing factors that may influence healthcare utilization and population
health management.

From a population health perspective, understanding both medical conditions and social risk factors
can help identify opportunities for preventive interventions, care coordination, and targeted
community support programs.


## Section 3 — Patient Utilization

### Business Question

Which patients demonstrate the highest levels of healthcare utilization, and what utilization patterns may indicate opportunities for care management or further clinical review?

### Display the Top Patients

In [17]:
top_utilizers = (
    patient_utilization
    .sort_values(
        "Total_Encounters",
        ascending=False,
    )
    .head(10)
)

display(top_utilizers)

,PATIENTID,Total_Encounters,First_Encounter,Last_Encounter,Ambulatory_Encounters,Emergency_Encounters,Hospice_Encounters,Inpatient_Encounters,Outpatient_Encounters,Snf_Encounters,Urgentcare_Encounters,Virtual_Encounters,Wellness_Encounters
0,4da67a54-6874-1651-2ad0-4ea849616d79,521,1950-08-26 16:40:01+00:00,2026-07-24 18:38:01+00:00,435,9,1,5,37,0,15,0,19
1,2710f015-2444-97bb-3625-02ba85e0176e,231,1967-08-14 20:36:30+00:00,2023-08-14 20:36:30+00:00,171,9,1,2,5,0,0,25,18
2,3f02e3fc-d09e-bef9-275d-07cc6dcbf61e,97,1967-08-14 20:36:30+00:00,2026-06-20 17:06:24+00:00,54,11,1,7,5,0,5,0,14
3,acaf231b-0183-8f6f-a90e-379711d23bf7,83,1964-07-27 20:36:30+00:00,2017-12-04 20:36:30+00:00,47,8,0,4,4,0,3,0,17
4,eb1db751-6d6c-bcc2-c93f-8b7b6f6c4d6a,79,1965-08-02 20:36:30+00:00,2001-04-23 20:36:30+00:00,46,16,0,3,1,0,0,0,13
5,04848ab2-ab54-ffc6-35b7-ced4cea9ac62,79,1967-08-14 20:36:30+00:00,2011-07-18 20:36:30+00:00,47,9,1,3,1,0,2,0,16
6,78ae2116-7a07-e8df-8cb2-524af4ab3dc1,66,1967-08-14 20:36:30+00:00,2003-04-21 20:36:30+00:00,39,7,0,3,3,0,1,0,13
7,b3b4cc1b-48fc-d89d-36a1-c583ec29baa1,55,1965-08-02 20:36:30+00:00,1998-12-28 20:36:30+00:00,22,17,0,1,2,0,1,0,12
8,a04ac436-c736-5589-db9c-599249ba68c8,54,1984-11-12 17:11:01+00:00,2026-06-23 17:11:01+00:00,25,2,0,1,6,0,6,0,14
9,f9e60e09-cacc-b2c5-8445-b5e7cd70ed9b,49,1988-08-31 18:44:00+00:00,2025-09-24 18:44:00+00:00,27,0,0,0,9,0,0,0,13


### Visualization

In [ ]:
fig = px.bar(
    top_utilizers.sort_values(
        "Total_Encounters",
        ascending=True,
    ),
    x="Total_Encounters",
    y="PATIENTID",
    orientation="h",
    title="Patients with the Highest Healthcare Utilization",
    labels={
        "PATIENTID": "Patient",
        "Total_Encounters": "Number of Encounters",
    },
    text="Total_Encounters",
)

fig.update_layout(
    showlegend=False,
    yaxis_title="Patient",
    xaxis_title="Total Encounters",
)

fig.show()

### Dynamic Consultant Interpretation

In [ ]:
from IPython.display import Markdown, display

highest = top_utilizers.iloc[0]
average = patient_utilization["Total_Encounters"].mean()

display(
    Markdown(
f"""
### Consultant Interpretation

The highest healthcare utilizer was **Patient {highest['PATIENTID']}**, with
**{highest['Total_Encounters']} encounters**, compared with a population average of
**{average:.1f} encounters per patient**.

Patients with substantially higher encounter volumes may represent individuals with
complex medical needs, chronic disease burden, or increased reliance on healthcare
services. These patients are potential candidates for additional care management,
case management, or longitudinal clinical review.

Understanding utilization patterns helps healthcare organizations prioritize
high-need populations, allocate clinical resources effectively, and identify
opportunities to reduce avoidable healthcare utilization while improving continuity
of care.
"""
    )
)


### Consultant Interpretation

The highest healthcare utilizer was **Patient 4da67a54-6874-1651-2ad0-4ea849616d79**, with
**521 encounters**, compared with a population average of
**93.4 encounters per patient**.

Patients with substantially higher encounter volumes may represent individuals with
complex medical needs, chronic disease burden, or increased reliance on healthcare
services. These patients are potential candidates for additional care management,
case management, or longitudinal clinical review.

Understanding utilization patterns helps healthcare organizations prioritize
high-need populations, allocate clinical resources effectively, and identify
opportunities to reduce avoidable healthcare utilization while improving continuity
of care.


## Section 4 — Population Risk Stratification

### Business Question

Which patients are at elevated risk based on their clinical burden, healthcare utilization, and healthcare spending, and how can the population be prioritized for targeted interventions?

### Build the Population Profile

In [18]:
patient_costs = calculate_patient_costs(
    claims_transactions=claims_transactions,
    patients=patients,
)

display(patient_costs.head())

,PATIENTID,Total_Charges,Number_of_Claims,Charge_Transactions,Id,FIRST,LAST,GENDER,BIRTHDATE,Patient
0,4da67a54-6874-1651-2ad0-4ea849616d79,654932.51,1123,2012,4da67a54-6874-1651-2ad0-4ea849616d79,Alonso270,Doyle959,M,1948-07-24,Patient 001
1,04848ab2-ab54-ffc6-35b7-ced4cea9ac62,446026.94,133,313,04848ab2-ab54-ffc6-35b7-ced4cea9ac62,Fausto876,Nader710,M,1949-06-20,Patient 002
2,2710f015-2444-97bb-3625-02ba85e0176e,335889.95,291,557,2710f015-2444-97bb-3625-02ba85e0176e,Dennis979,Zulauf375,M,1949-06-20,Patient 003
3,461c6b9d-81b3-3ff7-9463-bec1b2e0f5b8,235632.59,46,157,461c6b9d-81b3-3ff7-9463-bec1b2e0f5b8,Tillie335,Dach178,F,1997-06-07,Patient 004
4,a04ac436-c736-5589-db9c-599249ba68c8,195565.68,99,319,a04ac436-c736-5589-db9c-599249ba68c8,Alicia629,Fisher429,F,1981-11-17,Patient 005


In [19]:
population_profile = build_patient_population_profile(
    patients=patients,
    condition_burden=patient_condition_burden,
    patient_utilization=patient_utilization,
    patient_costs=patient_costs,
)
display(population_profile.head())


,PATIENTID,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,MIDDLE,LAST,...,Emergency_Encounters,Hospice_Encounters,Inpatient_Encounters,Outpatient_Encounters,Snf_Encounters,Urgentcare_Encounters,Virtual_Encounters,Wellness_Encounters,Total_Charges,Number_of_Claims
0,d994c47b-2b03-c7c2-b0d9-502d52408e4e,2021-05-13,NaN,999-11-4073,NaN,NaN,NaN,Terica746,Jesenia223,Jacobs452,...,0,0,0,2,0,1,0,14,19019.10,24
1,461c6b9d-81b3-3ff7-9463-bec1b2e0f5b8,1997-06-07,NaN,999-48-9983,S99912352,X42544153X,Mrs.,Tillie335,NaN,Dach178,...,2,0,0,11,0,0,0,6,235632.59,46
2,b1c9fdd3-a94e-8854-5f7f-97181b70fe00,1999-06-01,NaN,999-99-3428,S99936422,X31594479X,Mr.,Cesar434,Ferdinand55,Williamson769,...,1,0,1,2,1,0,0,4,43832.43,16
3,f9e60e09-cacc-b2c5-8445-b5e7cd70ed9b,1970-07-08,NaN,999-55-1414,S99944607,X47493840X,Mrs.,Jaclyn627,Roslyn469,Hessel84,...,0,0,0,9,0,0,0,13,85037.07,102
4,a04ac436-c736-5589-db9c-599249ba68c8,1981-11-17,NaN,999-15-1099,S99949400,X83185826X,Mrs.,Alicia629,NaN,Fisher429,...,2,0,1,6,0,6,0,14,195565.68,99


### Calculate Risk Scores

In [20]:
population_risk = stratify_population_risk(
    population_profile
)

### View Highest-Risk Patients

In [21]:
high_risk_patients = (
    population_risk
    .sort_values(
        "Risk_Score",
        ascending=False,
    )
    .head(10)
)

display(
    high_risk_patients[
        [
            "PATIENTID",
            "Risk_Score",
            "Risk_Tier",
            "Total_Charges",
            "Total_Encounters",
            "Unique_Conditions",
        ]
    ]
)

,PATIENTID,Risk_Score,Risk_Tier,Total_Charges,Total_Encounters,Unique_Conditions
0,4da67a54-6874-1651-2ad0-4ea849616d79,1.000000,High,654932.51,521,49
1,2710f015-2444-97bb-3625-02ba85e0176e,0.850000,High,335889.95,231,28
2,04848ab2-ab54-ffc6-35b7-ced4cea9ac62,0.845312,High,446026.94,79,32
3,3f02e3fc-d09e-bef9-275d-07cc6dcbf61e,0.790625,High,192642.41,97,37
4,acaf231b-0183-8f6f-a90e-379711d23bf7,0.731250,Moderate,193037.28,83,28
5,b3b4cc1b-48fc-d89d-36a1-c583ec29baa1,0.625000,Moderate,129114.71,55,29
6,eb1db751-6d6c-bcc2-c93f-8b7b6f6c4d6a,0.568750,Moderate,122616.88,79,25
7,a04ac436-c736-5589-db9c-599249ba68c8,0.553125,Moderate,195565.68,54,21
8,461c6b9d-81b3-3ff7-9463-bec1b2e0f5b8,0.481250,Low,235632.59,39,14
9,78ae2116-7a07-e8df-8cb2-524af4ab3dc1,0.435937,Low,71086.30,66,25


### Risk Tier Distribution

In [22]:
risk_summary = (
    population_risk["Risk_Tier"]
    .value_counts()
    .rename_axis("Risk_Tier")
    .reset_index(name="Patients")
)

display(risk_summary)

,Risk_Tier,Patients
0,Low,8
1,High,4
2,Moderate,4


### Visualization

In [23]:
fig = px.bar(
    risk_summary,
    x="Risk_Tier",
    y="Patients",
    text="Patients",
    title="Population Risk Tier Distribution",
)

fig.update_layout(
    xaxis_title="Risk Tier",
    yaxis_title="Number of Patients",
    showlegend=False,
)

fig.show()

### Dynamic Consultant Interpretation

In [24]:
from IPython.display import Markdown, display

highest = high_risk_patients.iloc[0]

high_count = (
    population_risk["Risk_Tier"] == "High"
).sum()

display(
    Markdown(
f"""
### Consultant Interpretation

The rule-based population health model classified **{high_count} of {len(population_risk)} patients**
as **High Risk** based on a composite assessment of healthcare spending, encounter utilization,
and clinical condition burden.

The highest-risk patient (**{highest['PATIENTID']}**) achieved a composite risk score of
**{highest['Risk_Score']:.2f}**, supported by:

- **{highest['Total_Charges']:,.0f}** in total healthcare charges
- **{highest['Total_Encounters']}** healthcare encounters
- **{highest['Unique_Conditions']}** unique documented conditions

Rather than relying on a single metric, this composite score integrates financial,
utilization, and clinical factors to prioritize patients who may benefit most from
proactive care management, chronic disease programs, or multidisciplinary interventions.

*Note: This transparent rule-based model is intended for demonstration purposes and
does not represent a validated clinical risk prediction model.*
"""
    )
)


### Consultant Interpretation

The rule-based population health model classified **4 of 16 patients**
as **High Risk** based on a composite assessment of healthcare spending, encounter utilization,
and clinical condition burden.

The highest-risk patient (**4da67a54-6874-1651-2ad0-4ea849616d79**) achieved a composite risk score of
**1.00**, supported by:

- **654,933** in total healthcare charges
- **521** healthcare encounters
- **49** unique documented conditions

Rather than relying on a single metric, this composite score integrates financial,
utilization, and clinical factors to prioritize patients who may benefit most from
proactive care management, chronic disease programs, or multidisciplinary interventions.

*Note: This transparent rule-based model is intended for demonstration purposes and
does not represent a validated clinical risk prediction model.*


## Section 5 — Risk Tier Profile Analysis

### Business Question

How do healthcare utilization, clinical burden, and financial cost differ across the population risk tiers?

### summary table

In [25]:
risk_profile_summary = (
    population_risk
    .groupby("Risk_Tier", as_index=False)
    .agg(
        Patients=("PATIENTID", "count"),
        Average_Charges=("Total_Charges", "mean"),
        Average_Encounters=("Total_Encounters", "mean"),
        Average_Conditions=("Unique_Conditions", "mean"),
        Average_Claims=("Number_of_Claims", "mean"),
        Average_Risk_Score=("Risk_Score", "mean"),
    )
)

display(risk_profile_summary.round(2))

,Risk_Tier,Patients,Average_Charges,Average_Encounters,Average_Conditions,Average_Claims,Average_Risk_Score
0,High,4,407372.95,232.00,36.50,431.00,0.87
1,Low,8,84622.43,36.88,19.12,57.88,0.32
2,Moderate,4,160083.64,67.75,25.75,94.75,0.62


### Visualization 1 — Average Charges by Risk Tier

In [26]:
fig = px.bar(
    risk_profile_summary,
    x="Risk_Tier",
    y="Average_Charges",
    text="Average_Charges",
    title="Average Total Charges by Risk Tier",
    labels={
        "Risk_Tier": "Risk Tier",
        "Average_Charges": "Average Charges ($)"
    },
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside",
)

fig.update_layout(showlegend=False)

fig.show()

### Visualization 2 — Average Encounters by Risk Tier

In [27]:
fig = px.bar(
    risk_profile_summary,
    x="Risk_Tier",
    y="Average_Encounters",
    text="Average_Encounters",
    title="Average Healthcare Encounters by Risk Tier",
)

fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside",
)

fig.update_layout(showlegend=False)

fig.show()

### Dynamic interpretation

In [28]:
highest = risk_profile_summary.loc[
    risk_profile_summary["Average_Charges"].idxmax()
]

lowest = risk_profile_summary.loc[
    risk_profile_summary["Average_Charges"].idxmin()
]

print(
    f"The {highest['Risk_Tier']} group had the highest average healthcare charges "
    f"(${highest['Average_Charges']:,.0f}) and averaged "
    f"{highest['Average_Encounters']:.1f} healthcare encounters and "
    f"{highest['Average_Conditions']:.1f} documented conditions per patient."
)

print()

print(
    f"In comparison, the {lowest['Risk_Tier']} group demonstrated substantially "
    f"lower utilization and healthcare costs, suggesting that the rule-based "
    f"risk stratification appropriately separates lower- and higher-resource "
    f"patient populations."
)

The High group had the highest average healthcare charges ($407,373) and averaged 232.0 healthcare encounters and 36.5 documented conditions per patient.

In comparison, the Low group demonstrated substantially lower utilization and healthcare costs, suggesting that the rule-based risk stratification appropriately separates lower- and higher-resource patient populations.


### Consultant Interpretation

The population-level summary demonstrates clear differences across the risk tiers. As expected, patients classified as higher risk exhibit greater healthcare utilization, higher total charges, more documented clinical conditions, and increased claim volume.

This type of segmentation enables healthcare organizations to prioritize limited care-management resources toward patients with the greatest expected healthcare needs while allowing lower-risk populations to receive routine preventive and primary care services.

Although this portfolio project uses a transparent rule-based model, the workflow reflects a common population health analytics approach used by payers, accountable care organizations (ACOs), and integrated delivery networks to identify patient cohorts for targeted intervention.

## Section 6: Population Health Opportunities

### Business Question

Which patient populations may benefit from targeted interventions based on healthcare utilization, clinical burden, and healthcare spending?

### Opportunity 1 — High Utilization Patients

In [29]:
high_utilizers = (
    population_risk
    .sort_values(
        "Total_Encounters",
        ascending=False,
    )
    .head(10)
)

display(
    high_utilizers[
        [
            "PATIENTID",
            "Risk_Tier",
            "Total_Encounters",
            "Unique_Conditions",
            "Total_Charges",
            "Risk_Score",
        ]
    ]
)

,PATIENTID,Risk_Tier,Total_Encounters,Unique_Conditions,Total_Charges,Risk_Score
0,4da67a54-6874-1651-2ad0-4ea849616d79,High,521,49,654932.51,1.000000
1,2710f015-2444-97bb-3625-02ba85e0176e,High,231,28,335889.95,0.850000
3,3f02e3fc-d09e-bef9-275d-07cc6dcbf61e,High,97,37,192642.41,0.790625
4,acaf231b-0183-8f6f-a90e-379711d23bf7,Moderate,83,28,193037.28,0.731250
2,04848ab2-ab54-ffc6-35b7-ced4cea9ac62,High,79,32,446026.94,0.845312
6,eb1db751-6d6c-bcc2-c93f-8b7b6f6c4d6a,Moderate,79,25,122616.88,0.568750
9,78ae2116-7a07-e8df-8cb2-524af4ab3dc1,Low,66,25,71086.30,0.435937
5,b3b4cc1b-48fc-d89d-36a1-c583ec29baa1,Moderate,55,29,129114.71,0.625000
7,a04ac436-c736-5589-db9c-599249ba68c8,Moderate,54,21,195565.68,0.553125
12,f9e60e09-cacc-b2c5-8445-b5e7cd70ed9b,Low,49,24,85037.07,0.396875


### Opportunity 2 — High Clinical Burden

In [30]:
complex_patients = (
    population_risk
    .sort_values(
        "Unique_Conditions",
        ascending=False,
    )
    .head(10)
)

display(
    complex_patients[
        [
            "PATIENTID",
            "Risk_Tier",
            "Unique_Conditions",
            "Total_Encounters",
            "Total_Charges",
            "Risk_Score",
        ]
    ]
)

,PATIENTID,Risk_Tier,Unique_Conditions,Total_Encounters,Total_Charges,Risk_Score
0,4da67a54-6874-1651-2ad0-4ea849616d79,High,49,521,654932.51,1.000000
3,3f02e3fc-d09e-bef9-275d-07cc6dcbf61e,High,37,97,192642.41,0.790625
2,04848ab2-ab54-ffc6-35b7-ced4cea9ac62,High,32,79,446026.94,0.845312
5,b3b4cc1b-48fc-d89d-36a1-c583ec29baa1,Moderate,29,55,129114.71,0.625000
1,2710f015-2444-97bb-3625-02ba85e0176e,High,28,231,335889.95,0.850000
4,acaf231b-0183-8f6f-a90e-379711d23bf7,Moderate,28,83,193037.28,0.731250
10,b8cecc26-d9d9-1355-3714-7dc5a4f1a6b4,Low,28,46,73393.97,0.428125
11,87de5b32-b3d1-455c-4121-08d8ef37f1fe,Low,27,32,102206.34,0.403125
9,78ae2116-7a07-e8df-8cb2-524af4ab3dc1,Low,25,66,71086.30,0.435937
6,eb1db751-6d6c-bcc2-c93f-8b7b6f6c4d6a,Moderate,25,79,122616.88,0.568750


### Opportunity 3 — Highest Healthcare Costs

In [31]:
highest_cost = (
    population_risk
    .sort_values(
        "Total_Charges",
        ascending=False,
    )
    .head(10)
)

display(
    highest_cost[
        [
            "PATIENTID",
            "Risk_Tier",
            "Total_Charges",
            "Total_Encounters",
            "Unique_Conditions",
            "Risk_Score",
        ]
    ]
)

,PATIENTID,Risk_Tier,Total_Charges,Total_Encounters,Unique_Conditions,Risk_Score
0,4da67a54-6874-1651-2ad0-4ea849616d79,High,654932.51,521,49,1.000000
2,04848ab2-ab54-ffc6-35b7-ced4cea9ac62,High,446026.94,79,32,0.845312
1,2710f015-2444-97bb-3625-02ba85e0176e,High,335889.95,231,28,0.850000
8,461c6b9d-81b3-3ff7-9463-bec1b2e0f5b8,Low,235632.59,39,14,0.481250
7,a04ac436-c736-5589-db9c-599249ba68c8,Moderate,195565.68,54,21,0.553125
4,acaf231b-0183-8f6f-a90e-379711d23bf7,Moderate,193037.28,83,28,0.731250
3,3f02e3fc-d09e-bef9-275d-07cc6dcbf61e,High,192642.41,97,37,0.790625
5,b3b4cc1b-48fc-d89d-36a1-c583ec29baa1,Moderate,129114.71,55,29,0.625000
6,eb1db751-6d6c-bcc2-c93f-8b7b6f6c4d6a,Moderate,122616.88,79,25,0.568750
11,87de5b32-b3d1-455c-4121-08d8ef37f1fe,Low,102206.34,32,27,0.403125


### Opportunity Summary

In [32]:
opportunity_summary = pd.DataFrame(
    {
        "Opportunity": [
            "High Utilization",
            "High Clinical Burden",
            "High Healthcare Cost",
        ],
        "Patients Reviewed": [10, 10, 10],
        "Primary Driver": [
            "Frequent healthcare encounters",
            "Multiple documented conditions",
            "High healthcare expenditures",
        ],
        "Potential Intervention": [
            "Care coordination",
            "Chronic disease management",
            "Cost/utilization review",
        ],
    }
)

display(opportunity_summary)

,Opportunity,Patients Reviewed,Primary Driver,Potential Intervention
0,High Utilization,10,Frequent healthcare encounters,Care coordination
1,High Clinical Burden,10,Multiple documented conditions,Chronic disease management
2,High Healthcare Cost,10,High healthcare expenditures,Cost/utilization review


### Dynamic Interpretation

In [33]:
highest_utilizer = high_utilizers.iloc[0]
highest_complexity = complex_patients.iloc[0]
highest_spender = highest_cost.iloc[0]

print(
    f"The highest-utilization patient recorded "
    f"{highest_utilizer['Total_Encounters']} healthcare encounters."
)

print(
    f"The patient with the greatest clinical burden had "
    f"{highest_complexity['Unique_Conditions']} documented conditions."
)

print(
    f"The highest-cost patient generated "
    f"${highest_spender['Total_Charges']:,.0f} in healthcare charges."
)

The highest-utilization patient recorded 521 healthcare encounters.
The patient with the greatest clinical burden had 49 documented conditions.
The highest-cost patient generated $654,933 in healthcare charges.


### Consultant Interpretation

This analysis highlights three patient cohorts that commonly receive focused attention in population health programs.

- Patients with high healthcare utilization may benefit from care coordination, discharge planning, or improved access to primary care to reduce avoidable utilization.
- Patients with high clinical burden often require multidisciplinary management, medication reconciliation, and chronic disease monitoring.
- Patients with the highest healthcare expenditures represent an opportunity to review drivers of cost, evaluate care pathways, and identify opportunities for improved efficiency while maintaining quality of care.

These cohorts are not mutually exclusive. Many patients appear in multiple categories, demonstrating that healthcare complexity frequently arises from the combined effects of chronic disease burden, frequent healthcare utilization, and elevated costs. Population health analytics enables healthcare organizations to identify these patients proactively and allocate care-management resources where they may have the greatest impact.

## Section 7 — Care Management Target Cohorts

### Business Question

Which patients may benefit from different population health interventions based on their utilization, clinical complexity, and healthcare spending patterns?

### Cohort 1 — Complex, High-Cost Patients

In [34]:
high_cost_threshold = population_risk["Total_Charges"].quantile(0.90)
high_condition_threshold = population_risk["Unique_Conditions"].quantile(0.90)

complex_high_cost = population_risk[
    (population_risk["Total_Charges"] >= high_cost_threshold)
    &
    (population_risk["Unique_Conditions"] >= high_condition_threshold)
]

display(
    complex_high_cost[
        [
            "PATIENTID",
            "Risk_Tier",
            "Total_Charges",
            "Unique_Conditions",
            "Total_Encounters",
            "Risk_Score",
        ]
    ]
)

,PATIENTID,Risk_Tier,Total_Charges,Unique_Conditions,Total_Encounters,Risk_Score
0,4da67a54-6874-1651-2ad0-4ea849616d79,High,654932.51,49,521,1.0


### Cohort 2 — Frequent Healthcare Users

In [35]:
high_utilization_threshold = population_risk["Total_Encounters"].quantile(0.90)

frequent_users = population_risk[
    population_risk["Total_Encounters"] >= high_utilization_threshold
]

display(
    frequent_users[
        [
            "PATIENTID",
            "Risk_Tier",
            "Total_Encounters",
            "Total_Charges",
            "Unique_Conditions",
        ]
    ]
)

,PATIENTID,Risk_Tier,Total_Encounters,Total_Charges,Unique_Conditions
0,4da67a54-6874-1651-2ad0-4ea849616d79,High,521,654932.51,49
1,2710f015-2444-97bb-3625-02ba85e0176e,High,231,335889.95,28


### Cohort 3 — Rising-Risk Patients

In [36]:
rising_risk = population_risk[
    (population_risk["Risk_Tier"] == "Moderate Risk")
    &
    (
        population_risk["Unique_Conditions"]
        >= population_risk["Unique_Conditions"].median()
    )
]

display(
    rising_risk[
        [
            "PATIENTID",
            "Risk_Tier",
            "Unique_Conditions",
            "Total_Encounters",
            "Total_Charges",
            "Risk_Score",
        ]
    ]
    .head(15)
)

,PATIENTID,Risk_Tier,Unique_Conditions,Total_Encounters,Total_Charges,Risk_Score


### Executive Summary Table

In [37]:
cohort_summary = pd.DataFrame(
    {
        "Target Cohort": [
            "Complex High-Cost",
            "Frequent Healthcare Users",
            "Rising-Risk Patients",
        ],
        "Patients": [
            len(complex_high_cost),
            len(frequent_users),
            len(rising_risk),
        ],
        "Potential Intervention": [
            "Comprehensive care management",
            "Care coordination",
            "Preventive outreach",
        ],
    }
)

display(cohort_summary)

,Target Cohort,Patients,Potential Intervention
0,Complex High-Cost,1,Comprehensive care management
1,Frequent Healthcare Users,2,Care coordination
2,Rising-Risk Patients,0,Preventive outreach


### Visualization

In [38]:
fig = px.bar(
    cohort_summary,
    x="Target Cohort",
    y="Patients",
    text="Patients",
    title="Population Health Target Cohorts",
)

fig.update_traces(textposition="outside")

fig.update_layout(
    showlegend=False,
    xaxis_title="Target Cohort",
    yaxis_title="Patients",
)

fig.show()

### Dynamic Interpretation

In [39]:
largest = cohort_summary.loc[
    cohort_summary["Patients"].idxmax()
]

print(
    f"The largest intervention cohort consists of "
    f"{largest['Patients']} patients identified as "
    f"{largest['Target Cohort']}. "
    f"These patients represent the greatest opportunity "
    f"for targeted population health interventions."
)

The largest intervention cohort consists of 2 patients identified as Frequent Healthcare Users. These patients represent the greatest opportunity for targeted population health interventions.


### Consultant Interpretation

Population health programs are most effective when interventions are tailored to the specific needs of different patient cohorts rather than applied uniformly across the entire population.

This analysis identifies three intervention groups. Complex high-cost patients may benefit from intensive multidisciplinary care management, frequent healthcare users may require improved care coordination to reduce avoidable utilization, and rising-risk patients represent an opportunity for preventive outreach before healthcare needs become more complex and costly.

Segmenting patients into actionable cohorts allows healthcare organizations to allocate limited clinical resources more efficiently while supporting improved quality of care and cost management.

## Section 8 — Executive Summary and Strategic Recommendations

### Business Question

Based on the population health analyses, what are the primary findings and what strategic actions should healthcare organizations consider?

### Executive Dashboard

In [40]:
executive_summary = pd.DataFrame(
    {
        "Metric": [
            "Total Patients",
            "Unique Conditions",
            "Total Healthcare Encounters",
            "Total Healthcare Charges",
            "High-Risk Patients",
        ],
        "Value": [
            len(population_risk),
            conditions["DESCRIPTION"].nunique(),
            int(population_risk["Total_Encounters"].sum()),
            f"${population_risk['Total_Charges'].sum():,.0f}",
            int(
                (
                    population_risk["Risk_Tier"] == "High Risk"
                ).sum()
            ),
        ],
    }
)

display(executive_summary)

,Metric,Value
0,Total Patients,16
1,Unique Conditions,102
2,Total Healthcare Encounters,1494
3,Total Healthcare Charges,"$2,946,806"
4,High-Risk Patients,0


### Executive Findings

In [41]:
top_condition = (
    condition_prevalence
    .sort_values(
        "Patients_With_Condition",
        ascending=False,
    )
    .iloc[0]
)

highest_risk = (
    population_risk
    .sort_values(
        "Risk_Score",
        ascending=False,
    )
    .iloc[0]
)

high_risk_count = (
    population_risk["Risk_Tier"]
    == "High Risk"
).sum()

high_risk_pct = (
    high_risk_count
    / len(population_risk)
    * 100
)

print("Executive Findings")
print("-" * 60)

print(
    f"• The most prevalent condition was "
    f"{top_condition['DESCRIPTION']} "
    f"affecting "
    f"{int(top_condition['Patients_With_Condition'])} patients."
)

print()

print(
    f"• The highest-risk patient accumulated "
    f"${highest_risk['Total_Charges']:,.0f} "
    f"in healthcare charges across "
    f"{int(highest_risk['Total_Encounters'])} encounters."
)

print()

print(
    f"• {high_risk_count} patients "
    f"({high_risk_pct:.1f}% of the population) "
    f"were classified as High Risk."
)

print()

print(
    "• Multiple intervention cohorts were identified based on "
    "healthcare utilization, clinical burden, and healthcare costs."
)

Executive Findings
------------------------------------------------------------
• The most prevalent condition was Medication review due (situation) affecting 16 patients.

• The highest-risk patient accumulated $654,933 in healthcare charges across 521 encounters.

• 0 patients (0.0% of the population) were classified as High Risk.

• Multiple intervention cohorts were identified based on healthcare utilization, clinical burden, and healthcare costs.


### Strategic Recommendations

In [42]:
recommendations = pd.DataFrame(
    {
        "Priority": [
            1,
            2,
            3,
            4,
        ],
        "Recommendation": [
            "Prioritize high-risk patients for comprehensive care management.",
            "Expand chronic disease management for clinically complex patients.",
            "Review high-cost utilization patterns to identify opportunities for care optimization.",
            "Implement proactive outreach for rising-risk patients to reduce future utilization.",
        ],
    }
)

display(recommendations)

,Priority,Recommendation
0,1,Prioritize high-risk patients for comprehensiv...
1,2,Expand chronic disease management for clinical...
2,3,Review high-cost utilization patterns to ident...
3,4,Implement proactive outreach for rising-risk p...


### Executive Interpretation

This portfolio demonstrates a complete population health analytics workflow beginning with claims and clinical data integration and progressing through disease prevalence analysis, utilization profiling, patient-level risk stratification, and identification of actionable intervention cohorts.

Several important patterns emerged:

- A relatively small proportion of patients accounted for the greatest healthcare complexity, utilization, and spending.
- Clinical burden, healthcare utilization, and financial cost frequently overlapped, supporting a multifactor approach to identifying patients who may benefit from targeted interventions.
- Population segmentation enabled the identification of cohorts appropriate for care coordination, chronic disease management, and preventive outreach.

Although this analysis uses synthetic Synthea data and a transparent rule-based risk stratification model, the analytical workflow reflects common approaches used by healthcare payers, accountable care organizations (ACOs), health systems, and population health consulting teams to support value-based care initiatives.

Future enhancements could incorporate validated predictive risk models, social determinants of health, pharmacy claims, laboratory data, and longitudinal utilization trends to further improve patient stratification and intervention planning.

## Conclusion

This project demonstrates the application of healthcare analytics, population health management, and business intelligence techniques using synthetic claims and clinical data. The analysis integrates reusable Python modules with interactive Plotly visualizations to answer practical healthcare business questions related to disease prevalence, healthcare utilization, patient risk, and intervention opportunities.

The notebook reflects a modular analytics workflow that emphasizes reproducibility, maintainability, and executive communication. While developed using synthetic data for educational purposes, the methods illustrated align with analytical approaches commonly used within payer organizations, health systems, accountable care organizations (ACOs), and healthcare consulting environments to support data-driven decision making.